# <div align="center">🌟 Madar (مدار) - Cloud LoRA Fine-Tuning Studio</div>
### <div align="center">Train LLMs natively in Ring/Madar with RingTensor & GPU Acceleration</div>

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/azzeddine2017/Madar/blob/main/notebooks/Madar_LoRA_Studio.ipynb) 
[![GitHub](https://img.shields.io/badge/GitHub-Madar-blue.svg)](https://github.com/azzeddine2017/Madar) 
[![RingTensor](https://img.shields.io/badge/Autograd-RingTensor%20C%20Kernels-green.svg)]() 
[![LoRA](https://img.shields.io/badge/Fine--Tuning-Low--Rank%20Adaptation-purple.svg)]()

---

Welcome to the **Madar LoRA Fine-Tuning Studio** on Google Colab.

This studio trains and fine-tunes LLMs (Qwen2.5 / LLaMA / BitNet) **100% natively in the Madar dialect** using the **Madar LoRA Engine** (`libraries/madar_ai/trainer/`) accelerated by **RingTensor C Autograd and OpenCL GPU kernels**.

### 🎯 Key Highlights:
- **Zero Python ML Overhead:** The forward pass, token masking, backward pass, and AdamW optimizer run directly in the Madar VM and C extensions.
- **Curated Ring Documentation Dataset:** Ingests **730 high-density instruction-response pairs** synthesized directly from the official Ring language Sphinx documentation.
- **Hardware Acceleration:** Native AVX2/FMA vectorization on CPU + OpenCL Zero-Copy GEMM on NVIDIA Tesla GPUs (T4 / V100 / A100).
- **Portable Checkpoints:** Exports lightweight standalone LoRA adapter weights (`.tensor` and `.json`) ready for deployment or local execution.

## ⚙️ Step 1: Hardware & Environment Verification
Verify the attached GPU accelerator (e.g. Tesla T4) and CPU SIMD features.

In [ ]:
# 1. Check NVIDIA GPU status
!nvidia-smi

# 2. Check CPU AVX2 & FMA instruction support
!lscpu | grep -E "Model name|avx2|fma"

## 🔨 Step 2: Clone Repository, Build Runtime & Extensions
Synchronizes the latest codebase, compiles the Core VM, and builds `libring_tensor.so` (with OpenCL) and `libring_cjson.so`.

In [ ]:
import os, sys

# 1. Clone repository if not present or pull latest updates
if not os.path.exists("/content/Madar/language"):
    !git clone https://github.com/azzeddine2017/Madar.git /content/Madar
    %cd /content/Madar
else:
    %cd /content/Madar
    !git fetch origin main
    !git reset --hard origin/main

# 2. Install essential packages for OpenCL, OpenMP, and GCC
!apt-get update -qq && apt-get install -y -qq build-essential g++ ocl-icd-opencl-dev opencl-headers curl libcurl4-openssl-dev

# 3. Execute the Linux build pipeline for Core Runtime & Extensions
!bash tools/build_linux.sh

# 4. Direct Verification & Path Alignment for Shared Libraries
!mkdir -p /content/Madar/lib /content/Madar/bin /content/Madar/extensions/ringtensor /content/Madar/extensions/ringcjson
!cp -f /content/Madar/lib/libring_tensor.so /content/Madar/bin/libring_tensor.so 2>/dev/null || true
!cp -f /content/Madar/lib/libring_tensor.so /content/Madar/extensions/ringtensor/libring_tensor.so 2>/dev/null || true
!cp -f /content/Madar/lib/libring_cjson.so /content/Madar/bin/libring_cjson.so 2>/dev/null || true
!cp -f /content/Madar/lib/libring_cjson.so /content/Madar/extensions/ringcjson/libring_cjson.so 2>/dev/null || true

# 5. Set runtime library paths
os.environ["LD_LIBRARY_PATH"] = "/content/Madar/lib:/content/Madar/bin:/content/Madar/extensions/ringtensor:/content/Madar/extensions/ringcjson:" + os.environ.get("LD_LIBRARY_PATH", "")

# 6. Verify library presence & compiler version
!ls -lh /content/Madar/lib/libring_tensor.so
!ls -lh /content/Madar/lib/libring_cjson.so
!bin/ring -v

## 📥 Step 3: Download Pre-Trained Base Model
Download the high-speed Qwen2.5-0.5B-Instruct GGUF base model for fine-tuning.

In [ ]:
!mkdir -p models
# Download base Qwen2.5 0.5B Instruct GGUF
!wget -c -q --show-progress -O models/qwen2.5-0.5b-instruct-q4_0.gguf \
    https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/qwen2.5-0.5b-instruct-q4_0.gguf

!ls -lh models/

## 📚 Step 4: Verify Instruction Training Dataset
Verify the synthesized 730 Ring documentation training pairs in `data/ring_instruct_dataset.jsonl`.

In [ ]:
# Verify dataset line count and file size
!wc -l data/ring_instruct_dataset.jsonl
!head -n 2 data/ring_instruct_dataset.jsonl

## 🪄 Step 5: Register Interactive `%%madar` Cell Magic
Allows executing Madar / Ring code directly within notebook cells with real-time streaming output.

In [ ]:
from IPython.core.magic import register_cell_magic
import subprocess, tempfile, os, sys

MADAR_BIN = os.path.abspath("bin/ring")

@register_cell_magic
def madar(line, cell):
    """IPython Cell Magic to execute Madar code with real-time streamed output."""
    with tempfile.NamedTemporaryFile("w", suffix=".ring", delete=False, encoding="utf-8") as f:
        f.write(cell)
        tmp_file = f.name
    try:
        env = os.environ.copy()
        env["LD_LIBRARY_PATH"] = "/content/Madar/lib:/content/Madar/bin:/content/Madar/extensions/ringtensor:/content/Madar/extensions/ringcjson:" + env.get("LD_LIBRARY_PATH", "")
        cwd = os.path.abspath("/content/Madar") if os.path.exists("/content/Madar") else os.path.abspath(".")
        
        proc = subprocess.Popen(
            [MADAR_BIN, tmp_file],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
            cwd=cwd
        )
        for out_line in proc.stdout:
            print(out_line, end="", flush=True)
        proc.wait()
    finally:
        if os.path.exists(tmp_file):
            os.remove(tmp_file)

ip = get_ipython()
ip.register_magic_function(madar, 'cell', 'ring')
print("✅ %%madar cell magic registered successfully with real-time output streaming!")

## 🔍 Step 6: Baseline Evaluation (Before Fine-Tuning)
Evaluate base model responses to Ring programming queries before applying LoRA.

In [ ]:
%%madar
load "madar_ai.ring"

? "=== Baseline Evaluation: Pre-Trained Model ==="
oModel = model_open("models/qwen2.5-0.5b-instruct-q4_0.gguf")
oModel.setTemperature(0.2)
oModel.setMaxTokens(128)

cQ = "Write a Ring code example to reverse a string and check if it is a palindrome."
cPrompt = oModel.formatChatPrompt(cQ, "You are an expert Ring language programmer.")
? oModel.generate(cPrompt, 120)


## 🚀 Step 7: Launch Full Madar LoRA Fine-Tuning
Train the LoRA adapters on all **730 instruction pairs** using **RingTensor C Autograd & AdamW** with Tesla T4 GPU acceleration.

In [ ]:
%%madar
load "madar_ai.ring"
load "madar_trainer.ring"

cModelPath = "models/qwen2.5-0.5b-instruct-q4_0.gguf"
cDataPath  = "data/ring_instruct_dataset.jsonl"

? "===================================================================="
? "  Madar AI - Full Cloud LoRA Training Execution (Colab GPU)           "
? "===================================================================="

# 1. Initialize Base Model with OpenCL GPU Acceleration
oModel = model_open(cModelPath)
oModel.setThreads(4)
oModel.setGPU(true)

# 2. Load the 730 Ring Instruction Pairs
oDataset = new MadarInstructDataset(cDataPath, "chatml")
? f"Loaded Dataset: {oDataset.size()} Instruction-Response Pairs"

# 3. Configure LoRA Trainer:
# Rank: 16 | Alpha: 32 | Base LR: 0.0002
oTrainer = new MadarLoRATrainer(oModel, oDataset, 16, 32.0, 0.0002)
oTrainer.nLogInterval  = 25
oTrainer.nSaveInterval = 100
oTrainer.cOutputDir    = "adapters/ring_lora_cloud"

# 4. Execute 3 Full Training Epochs (2190 Steps)
oTrainer.train(3)


## 📈 Step 8: Visualize Training Convergence
Plot the training loss curve across steps to inspect learning progress and convergence.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

csv_path = "adapters/ring_lora_cloud/final/loss_history.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    plt.figure(figsize=(10, 5))
    plt.plot(df["step"], df["loss"], color="#2E86AB", linewidth=1.5, label="Training Loss")
    # Rolling average
    if len(df) > 20:
        df["rolling"] = df["loss"].rolling(window=20).mean()
        plt.plot(df["step"], df["rolling"], color="#D90429", linewidth=2.0, label="Smoothed (20 steps)")
    plt.title("Madar LoRA Training Loss Convergence", fontsize=14, fontweight="bold")
    plt.xlabel("Step", fontsize=12)
    plt.ylabel("Loss", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ Loss history file will be available once training finishes.")

## 📦 Step 9: Package & Download Trained LoRA Adapters
Compress all trained LoRA weights (`.tensor` and `.json`) into a zip archive and download directly to your local machine.

In [ ]:
from google.colab import files

# Compress adapters
!zip -r ring_lora_cloud.zip adapters/ring_lora_cloud/final/

# Trigger browser download
files.download("ring_lora_cloud.zip")
print("✅ LoRA adapter weights packaged and ready for local deployment!")

## 💎 Step 10: Export Standalone Merged GGUF Model (Zero-Dependency)
Permanently fuse the trained LoRA attention weights directly into a single standalone `.gguf` file compatible with Madar, Ollama, and LM Studio.

In [ ]:
%%madar
load "madar_ai.ring"

? "===================================================================="
? "  Madar AI - Direct GGUF Model Exporter (Colab GPU)                  "
? "===================================================================="

cBaseModel = "models/qwen2.5-0.5b-instruct-q4_0.gguf"
cAdapter   = "adapters/ring_lora_cloud/final"
cOutGGUF   = "models/qwen2.5-0.5b-ring-instruct-merged.gguf"

oModel = model_open(cBaseModel)
oModel.loadLoRA(cAdapter)
oModel.setLoRAScale(0.005)

oModel.exportMergedGGUF(cOutGGUF)
oModel.close()
? "✅ Standalone GGUF model exported successfully: " + cOutGGUF


In [ ]:
from google.colab import files
import os

out_model = "models/qwen2.5-0.5b-ring-instruct-merged.gguf"
if os.path.exists(out_model):
    print(f"⬇️ Downloading standalone GGUF model ({os.path.getsize(out_model) / (1024*1024):.1f} MB)...")
    files.download(out_model)
else:
    print("❌ Exported GGUF file not found.")
